### Part D - Experiment 1: Inspect the Sparse Representation

In [1]:
import gzip
import json
import os

filepath = os.path.join("..", "data", "c4-train.00000-of-01024-30K.json.gz")
documents = []

with gzip.open(filepath, "rt", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        if 'text' in data:
            documents.append(data)
            
print("Number of documents: ",len(documents))

Number of documents:  30000


In [2]:
documents[0]

{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.',
 'timestamp': '2019-04-25T12:57:54Z',
 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}

In [3]:
texts = []
for doc in documents:
    texts.append(doc["text"])

print("Number of texts: ", len(texts))
print("First 500 texts: ")
print(texts[0][:500])

Number of texts:  30000
First 500 texts: 
Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat select


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_count = vectorizer.fit_transform(texts)

print("Vocabulary size:", len(vectorizer.vocabulary_))

N, V = X_count.shape

print("N =", N)
print("V =", V)
print("Matrix shape:", X_count.shape)

Vocabulary size: 193540
N = 30000
V = 193540
Matrix shape: (30000, 193540)


In [5]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer()

X_tfidf = tfidf_transformer.fit_transform(X_count)

print("TF-IDF matrix shape: ", X_tfidf.shape)

TF-IDF matrix shape:  (30000, 193540)


In [6]:
nnz = X_tfidf.nnz 
total_entries = N * V

sparsity = 1 - nnz / total_entries

print("Number of non-zero entries: ", nnz)
print("Total entries: ", total_entries)
print("Sparsity: ", sparsity)

Number of non-zero entries:  4985822
Total entries:  5806200000
Sparsity:  0.9991412934449382


Câu hỏi: Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều (V)?

Trả lời: Vì cần một không gian biểu diễn chung cho toàn bộ corpus để có thể so sánh các document với nhau.

In [9]:
# Top 20 terms phổ biến nhất (DF)
import numpy as np

feature_names = vectorizer.get_feature_names_out()
df_array = np.array((X_count > 0).sum(axis = 0)).flatten()

top_20_df = df_array.argsort()[::-1][:20]

for idx in top_20_df:
    print(f"{feature_names[idx]}: {df_array[idx]} docs")

the: 27893 docs
and: 27423 docs
to: 26689 docs
of: 26031 docs
a: 25905 docs
in: 25224 docs
for: 23651 docs
is: 22739 docs
with: 21405 docs
on: 20262 docs
that: 18370 docs
this: 17840 docs
are: 17594 docs
it: 17168 docs
s: 16959 docs
as: 16467 docs
at: 16347 docs
from: 16316 docs
be: 16153 docs
you: 16094 docs


In [13]:
# Top 20 terms có idf cao nhất 
idf_array = tfidf_transformer.idf_
top_20_idf = idf_array.argsort()[::-1][:20]

for idx in top_20_idf:
    print(f"{feature_names[idx]} : {idf_array[idx]}")

00000 : 10.615838812862137
𐌼𐌿𐌽𐌳𐍃 : 10.615838812862137
ﬂuid : 10.615838812862137
ﬂoors : 10.615838812862137
ﬂexibility : 10.615838812862137
ﬁxes : 10.615838812862137
ﬁt : 10.615838812862137
000000 : 10.615838812862137
000040 : 10.615838812862137
00005 : 10.615838812862137
0000856166 : 10.615838812862137
0001042 : 10.615838812862137
000116 : 10.615838812862137
00012 : 10.615838812862137
00015 : 10.615838812862137
00016 : 10.615838812862137
000165101 : 10.615838812862137
0002 : 10.615838812862137
00022 : 10.615838812862137
000226 : 10.615838812862137


In [ ]:
# Top 20 terms có TF-IDF cao nhất trong một document
doc_idx = 0
doc_tfidf = X_tfidf[doc_idx].toarray().flatten()
top_20_tfidf = doc_tfidf.argsort()[::-1][:20]

for idx in top_20_tfidf:
    if doc_tfidf[idx] > 0:
        print(f"{feature_names[idx]}: {doc_tfidf[idx]}")

bbq: 0.4607045414904368
class: 0.2751824296421107
meat: 0.19369418090937762
balay: 0.17897454277548658
kcbs: 0.172138725689874
lonestar: 0.1672886362820187
will: 0.1543062416916003
missoula: 0.1502338548530128
apron: 0.14101956555582323
smoker: 0.140154801680609
you: 0.13678897259769082
timelines: 0.1322309168016151
trimming: 0.13024518912385785
spectators: 0.1280525975090962
cost: 0.12760474312136075
rangers: 0.12470687357266719
22nd: 0.11927680365398317
beginner: 0.11699505689043391
beginners: 0.11616233458206372
culinary: 0.1157609061091688


Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?
- Dù một term có thể xuất hiện rất nhiều trong corpus nhưng nó không nhất thiết có TF-IDF cao. Do nếu xuất hiện nhiều => DF cao => IDF thấp nên TF-IDF thường không cao.

Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?
- Không vì TF-IDF = TF * IDF. Nếu term không xuất hiện trong document thì TF = 0 => TF-IDF = 0 dù DF rất cao. 